## Image Diff

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # note: also set this in the shell before launching jupyter
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
from collections import defaultdict
from standard_e2e import Modality

TRAIN_DIR     = '../data/processed/waymo_e2e/training/'
MANIFEST_PATH = '../data/train_manifest.json'
FEATURES_DIR  = '../data/processed/waymo_e2e/features/'

CLASSES = ['straight', 'left-turn', 'right-turn', 'lane-change-left', 'lane-change-right']

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)
print(f'{len(manifest)} sequences in manifest')

In [ ]:
# --- Confirm context-frame ordering: is the target the LAST element of context_fnames? ---
e = next(iter(manifest.values()))
print('target_fname:', e['target_fname'])
print('context last:', e['context_fnames'][-1], '(should == target)')
print('context prev:', e['context_fnames'][-2] if e['n_context'] > 1 else 'NONE')
assert e['context_fnames'][-1] == e['target_fname'], \
    'Expected target to be the last context frame; adjust prior-frame indexing below.'
print('\nConfirmed: prior frame = context_fnames[-2]')

## Per-frame functions (redefined for a self-contained notebook)

Copied from `waymo_feature_extraction.ipynb`. Keep in sync until factored into a shared module.

In [ ]:
# Exact copies of the v1 functions from feature_exploration_update.ipynb (cell 3).
# Keep in sync with that notebook until factored into a shared module.

def detect_edges_and_lines(img, canny_low=30, canny_high=100, hough_threshold=15,
                            min_line_length=20, max_line_gap=15, road_region_frac=0.55):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    H, W = gray.shape
    road_mask = np.zeros_like(gray); road_mask[int(H * road_region_frac):, :] = 1
    edges = cv2.Canny(gray, canny_low, canny_high)
    edges_masked = edges * road_mask
    lines_raw = cv2.HoughLinesP(edges_masked, rho=1, theta=np.pi / 180, threshold=hough_threshold,
                                 minLineLength=min_line_length, maxLineGap=max_line_gap)
    lines = []
    if lines_raw is not None:
        for line in lines_raw:
            x1, y1, x2, y2 = line[0]
            angle = abs(np.degrees(np.arctan2(y2 - y1, x2 - x1)))
            if 20 < angle < 75 or 105 < angle < 160:
                lines.append((x1, y1, x2, y2))
    return edges, np.array(lines)


def estimate_vanishing_point(lines, img_shape):
    H, W = img_shape[:2]
    def line_intersection(l1, l2):
        x1, y1, x2, y2 = l1; x3, y3, x4, y4 = l2
        denom = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
        if abs(denom) < 1e-6: return None
        t = ((x1 - x3) * (y3 - y4) - (y1 - y3) * (x3 - x4)) / denom
        return (x1 + t * (x2 - x1), y1 + t * (y2 - y1))
    if len(lines) < 2: return (0.5, 0.5), len(lines)
    pts = [p for i in range(len(lines)) for j in range(i + 1, len(lines))
           if (p := line_intersection(lines[i], lines[j])) and -W < p[0] < 2 * W and -H < p[1] < 2 * H]
    if not pts: return (0.5, 0.5), len(lines)
    xs, ys = np.array([p[0] for p in pts]), np.array([p[1] for p in pts])
    hist, xe, ye = np.histogram2d(xs, ys, bins=[np.linspace(-W, 2*W, 30), np.linspace(-H, 2*H, 30)])
    pi = np.unravel_index(hist.argmax(), hist.shape)
    return (((xe[pi[0]] + xe[pi[0]+1]) / 2) / W, ((ye[pi[1]] + ye[pi[1]+1]) / 2) / H), len(lines)

In [ ]:
# Exact copies of the v1 road functions from feature_exploration_update.ipynb (cell 3).

def get_robust_seed_color(hsv, H, W):
    pts = [(W//2, int(H*.95)), (W//2, int(H*.85)), (int(W*.35), int(H*.92)),
           (int(W*.65), int(H*.92)), (int(W*.35), int(H*.82)), (int(W*.65), int(H*.82))]
    colors, valid = [], []
    for sx, sy in pts:
        patch = hsv[max(0, sy-5):sy+5, max(0, sx-10):sx+10]
        if patch.size > 0:
            colors.append(np.mean(patch, axis=(0, 1))); valid.append((sx, sy))
    colors = np.array(colors)
    med = np.median(colors, axis=0)
    best = np.argmin(np.linalg.norm(colors - med, axis=1))
    return colors[best].astype(np.uint8), valid[best]


def segment_road(img, road_region_frac=0.55):
    H, W = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    seed_color, (sx, sy) = get_robust_seed_color(hsv, H, W)
    tol = np.array([20, 60, 60])
    mask = cv2.inRange(hsv, np.clip(seed_color.astype(int)-tol, 0, 255).astype(np.uint8),
                        np.clip(seed_color.astype(int)+tol, 0, 255).astype(np.uint8))
    mask[:int(H*road_region_frac), :] = 0
    flood = np.zeros((H+2, W+2), dtype=np.uint8)
    cv2.floodFill(mask.copy(), flood, (sx, sy), 255, loDiff=(10,10,10), upDiff=(10,10,10))
    road_mask = (flood[1:-1, 1:-1] * 255).astype(np.uint8)
    px = np.where(road_mask > 0)
    if len(px[0]) < 10:
        return road_mask, {'road_area_frac': 0., 'road_centroid_x': 0.5, 'road_taper': 0.}
    area = len(px[0]) / (H * W)
    cx = np.mean(px[1]) / W
    taper = (np.sum(road_mask[int(H*.9), :] > 0) - np.sum(road_mask[int(H*.7), :] > 0)) / W
    return road_mask, {'road_area_frac': area, 'road_centroid_x': cx, 'road_taper': taper}

 **Note:** the road/VP functions above are exact copies of the v1 functions from
`feature_exploration_update.ipynb` (cell 3), so `road_centroid_x`, `road_area_frac`, and
`vp_x` match the convention used by the committed `road.npy`. We use v1 (not v2) so the
difference feature is consistent with the existing baseline. Keep in sync until factored
into a shared module.

In [ ]:
from ultralytics import YOLO
yolo_model = YOLO('yolov8s.pt')

DRIVING_CLASSES = {0, 1, 2, 3, 5, 7, 9, 11}  # person, bicycle, car, motorcycle, bus, truck, light, sign

def yolo_lateral_aggregates(img, model, driving_classes=DRIVING_CLASSES):
    """Aggregate lateral detection signal over ALL driving-relevant detections
    (not per-class), designed to be differenced between frames.

    Returns dict:
      det_mean_x     — mean normalized center-x over all detections (0.5 if none)
      det_lr_balance — (left_count - right_count) / total  (0.0 if none)
    """
    H, W = img.shape[:2]
    results = model(img, verbose=False)
    xs = []
    for box in results[0].boxes:
        if int(box.cls) in driving_classes:
            x1, _, x2, _ = box.xyxy[0].tolist()
            xs.append(((x1 + x2) / 2) / W)
    if not xs:
        return {'det_mean_x': 0.5, 'det_lr_balance': 0.0}
    xs = np.array(xs)
    left = (xs < 0.5).sum()
    right = (xs >= 0.5).sum()
    return {
        'det_mean_x': float(xs.mean()),
        'det_lr_balance': float((left - right) / len(xs)),
    }

In [ ]:
# --- The 5 differenceable per-frame quantities ---
QUANTITY_KEYS = ['road_centroid_x', 'road_area_frac', 'vp_x', 'det_mean_x', 'det_lr_balance']

def compute_frame_quantities(img, model):
    """Compute the 5 scalar quantities for a single frame.
    Same function is reused across frames for the multi-frame (Stage 2) extension."""
    _, road = segment_road(img)
    _, lines = detect_edges_and_lines(img)
    (vp_x, _vp_y), _n = estimate_vanishing_point(lines, img.shape)
    det = yolo_lateral_aggregates(img, model)
    return {
        'road_centroid_x': road['road_centroid_x'],
        'road_area_frac':  road['road_area_frac'],
        'vp_x':            vp_x,
        'det_mean_x':      det['det_mean_x'],
        'det_lr_balance':  det['det_lr_balance'],
    }

def load_img(fname):
    data = np.load(os.path.join(TRAIN_DIR, fname), allow_pickle=True)
    return np.array(data['_modality_data'].item()[Modality.CAMERAS])

In [ ]:
# --- Limit-1 difference extraction over all sequences ---
# feature = Q(target) - Q(prior),  prior = context_fnames[-2]
# Sequences with n_context==1 have no prior frame -> zero vector, valid=False.
import time

framediff = []
valid_mask = []
labels = []
seq_ids = []

start = time.time()
for i, (sid, entry) in enumerate(manifest.items()):
    if (i + 1) % 100 == 0:
        el = time.time() - start
        print(f'  {i+1}/{len(manifest)} — {el/(i+1)*(len(manifest)-(i+1)):.0f}s remaining...')

    label = entry['label']
    if label not in CLASSES:
        continue  # skip the rare 'stationary'

    target_img = load_img(entry['target_fname'])
    q_target = compute_frame_quantities(target_img, yolo_model)

    if entry['n_context'] > 1:
        prior_fname = entry['context_fnames'][-2]
        prior_img = load_img(prior_fname)
        q_prior = compute_frame_quantities(prior_img, yolo_model)
        diff = [q_target[k] - q_prior[k] for k in QUANTITY_KEYS]
        valid = True
    else:
        diff = [0.0] * len(QUANTITY_KEYS)
        valid = False

    framediff.append(diff)
    valid_mask.append(valid)
    labels.append(label)
    seq_ids.append(sid)

framediff = np.array(framediff, dtype=np.float32)
valid_mask = np.array(valid_mask)
labels = np.array(labels)
seq_ids = np.array(seq_ids)

print(f'\nDone in {time.time()-start:.0f}s')
print(f'framediff shape: {framediff.shape}')
print(f'valid (has prior frame): {valid_mask.sum()} / {len(valid_mask)}')

In [ ]:
# --- Save (alongside existing arrays; does NOT overwrite anything) ---
np.save(os.path.join(FEATURES_DIR, 'framediff.npy'), framediff)
np.save(os.path.join(FEATURES_DIR, 'framediff_valid.npy'), valid_mask)
np.save(os.path.join(FEATURES_DIR, 'framediff_labels.npy'), labels)
np.save(os.path.join(FEATURES_DIR, 'framediff_seq_ids.npy'), seq_ids)
print('Saved framediff.npy (+ valid / labels / seq_ids)')

## Sanity Check

Before classifying, eyeball whether the lateral-difference quantities (`road_centroid_x`,
`det_mean_x`, `det_lr_balance`) show the expected directional pattern for lane changes vs.
straight. If lane-change-left/right separate on any of these, that's early evidence the
signal exists; if all classes overlap completely, the limit-1 step may be too small (→ Stage 2).

In [ ]:
# per-class mean of each difference dim (valid rows only)
v = valid_mask
print(f'{"quantity":<18} ' + '  '.join(f'{c[:10]:>10}' for c in CLASSES))
for j, k in enumerate(QUANTITY_KEYS):
    row = []
    for c in CLASSES:
        m = (labels == c) & v
        row.append(f'{framediff[m, j].mean():>+10.4f}')
    print(f'{k:<18} ' + '  '.join(row))

In [ ]:
# distribution plots: each difference dim, by class
fig, axes = plt.subplots(1, len(QUANTITY_KEYS), figsize=(4 * len(QUANTITY_KEYS), 3.5))
for j, (ax, k) in enumerate(zip(axes, QUANTITY_KEYS)):
    data_by_class = [framediff[(labels == c) & valid_mask, j] for c in CLASSES]
    ax.boxplot(data_by_class, tick_labels=[c[:6] for c in CLASSES], showfliers=False)
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.set_title(k, fontsize=10)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
plt.suptitle('Limit-1 frame-difference by maneuver class', fontsize=12)
plt.tight_layout()
plt.savefig('outputs/framediff_by_class.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- Stage 1 gate: does limit-1 framediff help lane-change recall? ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# use only valid rows (have a prior frame)
Xfd = framediff[valid_mask]
yfd = labels[valid_mask]

Xtr, Xte, ytr, yte = train_test_split(
    Xfd, yfd, test_size=0.25, random_state=0, stratify=yfd)
sc = StandardScaler().fit(Xtr)

print("=== framediff (limit-1) standalone ===")
for name, clf in [('SVM', SVC(kernel='rbf', random_state=0)),
                  ('RF', RandomForestClassifier(n_estimators=300, random_state=0))]:
    clf.fit(sc.transform(Xtr), ytr)
    pred = clf.predict(sc.transform(Xte))
    acc = (pred == yte).mean()
    # lane-change recall specifically
    print(f"\n{name}: acc={acc:.3f}")
    rep = classification_report(yte, pred, labels=CLASSES, output_dict=True, zero_division=0)
    for c in ['lane-change-left', 'lane-change-right']:
        print(f"   {c}: recall={rep[c]['recall']:.3f}")

In [ ]:
# --- Stage 2: multi-frame trend feature ---
# For each sequence, compute Q(frame) across ALL context frames, then fit a
# linear slope of each quantity vs. time. The slope captures sustained drift
# (e.g. a gradual lane change) that a single 250ms step misses.
import time

def fit_trend(frame_fnames, model):
    """Compute each quantity across frames, return per-quantity slope + last value.
    Slope is per-frame (dQ/frame); frames are 250ms apart."""
    Qs = []
    for fn in frame_fnames:
        img = load_img(fn)
        q = compute_frame_quantities(img, model)
        Qs.append([q[k] for k in QUANTITY_KEYS])
    Qs = np.array(Qs)                      # (n_frames, n_quant)
    n = len(Qs)
    if n < 2:
        # single frame: no trend, return zeros for slope + the value itself
        return np.concatenate([np.zeros(len(QUANTITY_KEYS)), Qs[0]])
    t = np.arange(n)
    # least-squares slope per quantity: cov(t,Q)/var(t)
    t_c = t - t.mean()
    slopes = (t_c[:, None] * (Qs - Qs.mean(0))).sum(0) / (t_c**2).sum()
    last = Qs[-1]                          # value at target frame
    return np.concatenate([slopes, last])  # 2*n_quant dims

# feature = [slope per quantity] + [target-frame value per quantity]
TREND_KEYS = [f'slope_{k}' for k in QUANTITY_KEYS] + [f'last_{k}' for k in QUANTITY_KEYS]

trend = []
trend_labels = []
trend_seq_ids = []
start = time.time()
for i, (sid, entry) in enumerate(manifest.items()):
    if (i + 1) % 100 == 0:
        el = time.time() - start
        print(f'  {i+1}/{len(manifest)} — {el/(i+1)*(len(manifest)-(i+1)):.0f}s remaining...')
    if entry['label'] not in CLASSES:
        continue
    feat = fit_trend(entry['context_fnames'], yolo_model)
    trend.append(feat)
    trend_labels.append(entry['label'])
    trend_seq_ids.append(sid)

trend = np.array(trend, dtype=np.float32)
trend_labels = np.array(trend_labels)
trend_seq_ids = np.array(trend_seq_ids)
print(f'\nDone in {time.time()-start:.0f}s. trend shape: {trend.shape}')

np.save(os.path.join(FEATURES_DIR, 'framediff_trend.npy'), trend)
np.save(os.path.join(FEATURES_DIR, 'framediff_trend_labels.npy'), trend_labels)
np.save(os.path.join(FEATURES_DIR, 'framediff_trend_seq_ids.npy'), trend_seq_ids)
print('Saved framediff_trend.npy')

In [ ]:
# --- Gate: does the multi-frame trend beat limit-1 on lane-change recall? ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# load if not in memory
trend = np.load(os.path.join(FEATURES_DIR, 'framediff_trend.npy'))
tlabels = np.load(os.path.join(FEATURES_DIR, 'framediff_trend_labels.npy'), allow_pickle=True)

Xtr, Xte, ytr, yte = train_test_split(
    trend, tlabels, test_size=0.25, random_state=0, stratify=tlabels)
sc = StandardScaler().fit(Xtr)

print("=== multi-frame trend feature ===")
for name, clf in [('SVM', SVC(kernel='rbf', random_state=0)),
                  ('RF', RandomForestClassifier(n_estimators=300, random_state=0))]:
    clf.fit(sc.transform(Xtr), ytr)
    pred = clf.predict(sc.transform(Xte))
    acc = (pred == yte).mean()
    rep = classification_report(yte, pred, labels=CLASSES, output_dict=True, zero_division=0)
    print(f"\n{name}: acc={acc:.3f}")
    for c in CLASSES:
        print(f"   {c:<20} recall={rep[c]['recall']:.3f}")

# confusion matrix for the better model (usually SVM here)
print("\nConfusion matrix (SVM):")
clf = SVC(kernel='rbf', random_state=0).fit(sc.transform(Xtr), ytr)
pred = clf.predict(sc.transform(Xte))
cm = confusion_matrix(yte, pred, labels=CLASSES)
print(f"{'':>18}" + ''.join(f'{c[:8]:>10}' for c in CLASSES))
for i, c in enumerate(CLASSES):
    print(f"{c:>18}" + ''.join(f'{cm[i,j]:>10}' for j in range(len(CLASSES))))

In [ ]:
# --- Diagnostic: lane-change failures, images + trajectory + feature values ---
import matplotlib.pyplot as plt
from standard_e2e import Modality, TrajectoryComponent

# refit trend model, capture which lane-change test sequences failed
from sklearn.model_selection import train_test_split
tr_idx = np.arange(len(trend))
Xtr, Xte, ytr, yte, itr, ite = train_test_split(
    trend, tlabels, tr_idx, test_size=0.25, random_state=0, stratify=tlabels)
sc = StandardScaler().fit(Xtr)
clf = SVC(kernel='rbf', random_state=0).fit(sc.transform(Xtr), ytr)
pred = clf.predict(sc.transform(Xte))

tseq = np.load(os.path.join(FEATURES_DIR, 'framediff_trend_seq_ids.npy'), allow_pickle=True)
test_seq = tseq[ite]

# lane-change sequences that were misclassified
lc_fails = [(test_seq[k], yte[k], pred[k]) for k in range(len(yte))
            if yte[k] in ('lane-change-left','lane-change-right') and pred[k] != yte[k]]
print(f'{len(lc_fails)} lane-change failures in test set')

def load_img(fn):
    d = np.load(os.path.join(TRAIN_DIR, fn), allow_pickle=True)
    return np.array(d['_modality_data'].item()[Modality.CAMERAS])

def diag_panel(sid, true, pred, n_frames=5):
    entry = manifest[sid]
    frames = entry['context_fnames']
    fidx = np.linspace(0, len(frames)-1, n_frames).astype(int)

    # trajectory from target
    d = np.load(os.path.join(TRAIN_DIR, entry['target_fname']), allow_pickle=True)
    fut = d['_modality_data'].item()[Modality.FUTURE_STATES]
    xs = fut.get(TrajectoryComponent.X).flatten(); ys = fut.get(TrajectoryComponent.Y).flatten()

    # road_centroid_x across the context window (the trend the feature saw)
    cxs = []
    for fn in frames:
        _, rf = segment_road(load_img(fn))
        cxs.append(rf['road_centroid_x'])

    fig, axes = plt.subplots(1, n_frames+2, figsize=((n_frames+2)*2.8, 2.8))
    for j, fi in enumerate(fidx):
        axes[j].imshow(load_img(frames[fi])); axes[j].axis('off'); axes[j].set_title(f'f{fi}', fontsize=8)
    # trajectory
    ax = axes[n_frames]
    ax.plot(-ys, xs, '-o', ms=2); ax.plot(-ys[0], xs[0],'go',ms=6); ax.plot(-ys[-1],xs[-1],'rs',ms=6)
    ax.axvline(0,color='gray',lw=.5,ls='--'); ax.set_aspect('equal','box')
    ax.set_title(f'traj lat={ys[-1]-ys[0]:+.1f}', fontsize=8); ax.tick_params(labelsize=6)
    # road_centroid_x trend
    ax2 = axes[n_frames+1]
    ax2.plot(cxs, '-o', ms=3); ax2.set_title('road_cx trend', fontsize=8)
    ax2.tick_params(labelsize=6)
    fig.suptitle(f'{sid[:8]}  TRUE {true} -> PRED {pred}', fontsize=10, y=1.05)
    plt.tight_layout(); plt.show()

# show first several failures
for sid, true, pred in lc_fails[:6]:
    diag_panel(sid, true, pred)

In [ ]:
import matplotlib.pyplot as plt
from standard_e2e import Modality, TrajectoryComponent

lat_by_class = {c: [] for c in ['lane-change-left','lane-change-right','straight']}
for sid, entry in manifest.items():
    if entry['label'] not in lat_by_class: continue
    d = np.load(os.path.join(TRAIN_DIR, entry['target_fname']), allow_pickle=True)
    fut = d['_modality_data'].item()[Modality.FUTURE_STATES]
    ys = fut.get(TrajectoryComponent.Y).flatten()
    lat_by_class[entry['label']].append(ys[-1]-ys[0])

fig, ax = plt.subplots(figsize=(9,4))
for c, vals in lat_by_class.items():
    ax.hist(np.abs(vals), bins=40, alpha=0.5, label=c, range=(0,10))
ax.axvline(2.5, color='red', ls='--', label='2.5m threshold')
ax.set_xlabel('|lateral displacement| (m)'); ax.legend(); ax.set_title('Lateral displacement by class')
plt.show()

In [ ]:
# --- Sanity check: is lateral displacement truncated/clipped, or is the 2.5m spike real? ---
from standard_e2e import Modality, TrajectoryComponent

# grab a handful of lane-change sequences
lc_sids = [sid for sid, e in manifest.items()
           if e['label'] in ('lane-change-left', 'lane-change-right')][:8]

print(f"{'seq':<10} {'label':<18} {'n_pts':>6} {'t_span':>7} {'|lat|':>7} {'|fwd|':>7} {'y[0]':>7} {'y[-1]':>7} {'max|y|':>7}")
print('-' * 90)
for sid in lc_sids:
    e = manifest[sid]
    d = np.load(os.path.join(TRAIN_DIR, e['target_fname']), allow_pickle=True)
    fut = d['_modality_data'].item()[Modality.FUTURE_STATES]
    xs = fut.get(TrajectoryComponent.X).flatten()
    ys = fut.get(TrajectoryComponent.Y).flatten()
    ts = fut.get(TrajectoryComponent.TIMESTAMP).flatten()
    valid = fut.get(TrajectoryComponent.IS_VALID).flatten()

    n_pts = len(ys)
    t_span = ts[-1] - ts[0]
    lat = ys[-1] - ys[0]
    fwd = xs[-1] - xs[0]
    print(f"{sid[:8]:<10} {e['label']:<18} {n_pts:>6} {t_span:>7.2f} "
          f"{abs(lat):>7.2f} {abs(fwd):>7.2f} {ys[0]:>7.2f} {ys[-1]:>7.2f} {np.abs(ys).max():>7.2f}")

# Now dump the FULL y-trajectory for the first two to see the shape
print("\n--- Full Y trajectory (lateral position over time) for 2 sequences ---")
for sid in lc_sids[:2]:
    e = manifest[sid]
    d = np.load(os.path.join(TRAIN_DIR, e['target_fname']), allow_pickle=True)
    fut = d['_modality_data'].item()[Modality.FUTURE_STATES]
    ys = fut.get(TrajectoryComponent.Y).flatten()
    valid = fut.get(TrajectoryComponent.IS_VALID).flatten()
    print(f"\n{sid[:8]} ({e['label']}):")
    print("  Y:", np.array2string(ys, precision=2, max_line_width=120))
    print("  valid:", valid.astype(int).tolist())

In [ ]:
# --- Test: are CLEAR lane changes (large |lat|) more separable than marginal ones? ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from standard_e2e import Modality, TrajectoryComponent
from collections import Counter 

# 1. lateral displacement per sequence, keyed by seq_id
lat_by_sid = {}
for sid, e in manifest.items():
    if e['label'] not in CLASSES:
        continue
    d = np.load(os.path.join(TRAIN_DIR, e['target_fname']), allow_pickle=True)
    ys = d['_modality_data'].item()[Modality.FUTURE_STATES].get(TrajectoryComponent.Y).flatten()
    lat_by_sid[sid] = abs(ys[-1] - ys[0])

# 2. train the trend model, tracking seq_ids into the test split
trend      = np.load(os.path.join(FEATURES_DIR, 'framediff_trend.npy'))
tlabels    = np.load(os.path.join(FEATURES_DIR, 'framediff_trend_labels.npy'), allow_pickle=True)
tseq       = np.load(os.path.join(FEATURES_DIR, 'framediff_trend_seq_ids.npy'), allow_pickle=True)

idx = np.arange(len(trend))
Xtr, Xte, ytr, yte, itr, ite = train_test_split(
    trend, tlabels, idx, test_size=0.25, random_state=0, stratify=tlabels)
sc = StandardScaler().fit(Xtr)
clf = RandomForestClassifier(n_estimators=300, random_state=0).fit(sc.transform(Xtr), ytr)
pred = clf.predict(sc.transform(Xte))
test_sids = tseq[ite]

# 3. among lane-change test sequences, split by |lat| magnitude and check recall
print("Lane-change recall, split by lateral displacement magnitude:")
print(f"{'|lat| bin':<14} {'n':>5} {'correct':>8} {'recall':>8}   {'where the misses went'}")
print('-' * 80)

bins = [(2.0, 2.75), (2.75, 3.5), (3.5, 5.0), (5.0, 50.0)]
for lc_class in ['lane-change-left', 'lane-change-right']:
    print(f"\n{lc_class}:")
    for lo, hi in bins:
        sel = [(k) for k in range(len(yte))
               if yte[k] == lc_class and lo <= lat_by_sid.get(test_sids[k], -1) < hi]
        if not sel:
            print(f"  {lo:.2f}-{hi:.1f}m   {'0':>5}   (none)")
            continue
        n = len(sel)
        correct = sum(pred[k] == yte[k] for k in sel)
        misses = Counter(pred[k] for k in sel if pred[k] != yte[k])
        miss_str = ', '.join(f'{v}->{p[:8]}' for p, v in misses.most_common())
        print(f"  {lo:.2f}-{hi:.1f}m {n:>5} {correct:>8} {correct/n:>8.2f}   {miss_str}")

In [ ]:
# --- Refined cheap check: lane-change PRECONDITION vs straight ---
# Filter to n_context>=10 (real looming window). Average first-third vs last-third
# of context frames to reduce per-frame detection noise. No tracking.
from standard_e2e import Modality
from collections import defaultdict
import numpy as np, random

VEHICLE_CLASSES = {2, 5, 7}  # car, bus, truck

def frame_signals(img, model):
    """Per-frame: largest central-forward vehicle box area, and left/right vehicle counts."""
    H, W = img.shape[:2]
    res = model(img, verbose=False)
    fwd_area = 0.0
    left_ct = right_ct = 0
    for b in res[0].boxes:
        if int(b.cls) not in VEHICLE_CLASSES:
            continue
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        cx = ((x1 + x2) / 2) / W
        area = ((x2 - x1) * (y2 - y1)) / (W * H)
        if 0.4 <= cx <= 0.6:
            fwd_area = max(fwd_area, area)
        if cx < 0.4: left_ct += 1
        if cx > 0.6: right_ct += 1
    return fwd_area, left_ct, right_ct

def load_img(fn):
    d = np.load(os.path.join(TRAIN_DIR, fn), allow_pickle=True)
    return np.array(d['_modality_data'].item()[Modality.CAMERAS])

def seq_precondition(frames, model):
    """Average signals over first-third vs last-third of the context window."""
    n = len(frames)
    k = max(1, n // 3)
    early = frames[:k]
    late = frames[-k:]
    def avg(fs):
        vals = [frame_signals(load_img(f), model) for f in fs]
        return np.mean([v[0] for v in vals]), np.mean([v[1] for v in vals]), np.mean([v[2] for v in vals])
    fa_e, l_e, r_e = avg(early)
    fa_l, l_l, r_l = avg(late)
    return {
        'fwd_present':  1.0 if fa_l > 0 else 0.0,       # vehicle ahead near prediction point
        'fwd_growth':   fa_l - fa_e,                     # looming: forward box grew?
        'left_clear':   1.0 if l_l == 0 else 0.0,        # left lane empty near prediction point
        'right_clear':  1.0 if r_l == 0 else 0.0,        # right lane empty
    }

# sample sequences per class, ONLY n_context >= 10
random.seed(0)
by_class = defaultdict(list)
for sid, e in manifest.items():
    if e['label'] in ('straight', 'lane-change-left', 'lane-change-right') and e['n_context'] >= 10:
        by_class[e['label']].append((sid, e))

print("Eligible (n_context>=10):", {c: len(v) for c, v in by_class.items()})
sample = {c: random.sample(v, min(60, len(v))) for c, v in by_class.items()}

rows = defaultdict(list)
for c, seqs in sample.items():
    for sid, e in seqs:
        rows[c].append(seq_precondition(e['context_fnames'], yolo_model))

order = ['straight', 'lane-change-left', 'lane-change-right']
print(f"\n{'signal':<14}" + ''.join(f'{c:>16}' for c in order))
for k in ['fwd_present', 'fwd_growth', 'left_clear', 'right_clear']:
    vals = [np.mean([r[k] for r in rows[c]]) for c in order]
    print(f"{k:<14}" + ''.join(f'{v:>16.3f}' for v in vals))

In [ ]:
# --- Feasibility check: can classical lane detection find lane lines on CLEAN frames? ---
# Not building a detector — just checking if lane lines are findable at all on best-case imagery.
import cv2, numpy as np, matplotlib.pyplot as plt
from standard_e2e import Modality

def load_img(fn):
    d = np.load(os.path.join(TRAIN_DIR, fn), allow_pickle=True)
    return np.array(d['_modality_data'].item()[Modality.CAMERAS])

def detect_lane_lines(img):
    """Standard classical lane pipeline. Returns overlay + line count."""
    H, W = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    # brighten check + edges
    edges = cv2.Canny(gray, 50, 150)
    # ROI: bottom 45% (road region), avoid the wide panoramic sides — keep central 60%
    mask = np.zeros_like(edges)
    y0 = int(H * 0.55)
    x0, x1 = int(W * 0.2), int(W * 0.8)
    mask[y0:, x0:x1] = 1
    edges_roi = edges * mask
    # Hough for line segments
    lines = cv2.HoughLinesP(edges_roi, 1, np.pi/180, threshold=20,
                            minLineLength=15, maxLineGap=20)
    overlay = img.copy()
    n_lane = 0
    if lines is not None:
        for l in lines:
            x1_, y1_, x2_, y2_ = l[0]
            ang = abs(np.degrees(np.arctan2(y2_ - y1_, x2_ - x1_)))
            # lane lines converging to VP: steep-ish, not horizontal
            if 25 < ang < 80:
                cv2.line(overlay, (x1_, y1_), (x2_, y2_), (0, 255, 0), 2)
                n_lane += 1
    return overlay, n_lane, edges_roi

# pick BEST-CASE frames: bright, straight-driving, daytime
# brightness proxy = mean pixel value; straight = label straight
import random; random.seed(3)
candidates = []
for sid, e in manifest.items():
    if e['label'] != 'straight':
        continue
    img = load_img(e['target_fname'])
    if img.mean() > 110:   # bright / daytime
        candidates.append((sid, e, img.mean()))
    if len(candidates) > 200:
        break

# take the brightest handful
candidates.sort(key=lambda c: -c[2])
picks = candidates[:6]
print(f"Testing {len(picks)} brightest daytime straight frames (mean brightness {picks[-1][2]:.0f}-{picks[0][2]:.0f})")

fig, axes = plt.subplots(len(picks), 2, figsize=(11, 3*len(picks)))
for i, (sid, e, br) in enumerate(picks):
    img = load_img(e['target_fname'])
    overlay, n, edges_roi = detect_lane_lines(img)
    axes[i,0].imshow(img); axes[i,0].set_title(f'{sid[:8]} (bright={br:.0f})', fontsize=9); axes[i,0].axis('off')
    axes[i,1].imshow(overlay); axes[i,1].set_title(f'{n} candidate lane lines', fontsize=9); axes[i,1].axis('off')
plt.suptitle('Lane detection feasibility on best-case (bright, straight) frames', fontsize=12)
plt.tight_layout()
plt.savefig('outputs/lane_feasibility.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# --- Test: does the central single-camera crop recover lane detection vs the panorama? ---
import cv2, numpy as np, matplotlib.pyplot as plt
from standard_e2e import Modality

def load_img(fn):
    d = np.load(os.path.join(TRAIN_DIR, fn), allow_pickle=True)
    return np.array(d['_modality_data'].item()[Modality.CAMERAS])

def detect_lines(img, x_lo_frac=0.2, x_hi_frac=0.8):
    """Same pipeline as before, ROI as fraction of THIS image's width."""
    H, W = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    mask = np.zeros_like(edges)
    y0 = int(H * 0.55)
    x0, x1 = int(W * x_lo_frac), int(W * x_hi_frac)
    mask[y0:, x0:x1] = 1
    edges_roi = edges * mask
    lines = cv2.HoughLinesP(edges_roi, 1, np.pi/180, threshold=20,
                            minLineLength=15, maxLineGap=20)
    overlay = img.copy(); n = 0
    if lines is not None:
        for l in lines:
            x1_, y1_, x2_, y2_ = l[0]
            ang = abs(np.degrees(np.arctan2(y2_ - y1_, x2_ - x1_)))
            if 25 < ang < 80:
                cv2.line(overlay, (x1_, y1_), (x2_, y2_), (0,255,0), 2)
                n += 1
    return overlay, n

# same bright straight frames as before
import random; random.seed(3)
candidates = []
for sid, e in manifest.items():
    if e['label'] != 'straight': continue
    img = load_img(e['target_fname'])
    if img.mean() > 110:
        candidates.append((sid, e, img.mean()))
    if len(candidates) > 200: break
candidates.sort(key=lambda c: -c[2])
picks = candidates[:5]

CENTER = (128, 256)  # front-center single-camera segment

fig, axes = plt.subplots(len(picks), 2, figsize=(11, 3*len(picks)))
for i, (sid, e, br) in enumerate(picks):
    img = load_img(e['target_fname'])
    # full panorama
    ov_full, n_full = detect_lines(img)
    # central crop — for a crop we can use the full crop width, less side masking
    crop = img[:, CENTER[0]:CENTER[1]]
    ov_crop, n_crop = detect_lines(crop, x_lo_frac=0.05, x_hi_frac=0.95)
    axes[i,0].imshow(ov_full); axes[i,0].set_title(f'{sid[:8]} PANORAMA: {n_full} lines', fontsize=9); axes[i,0].axis('off')
    axes[i,1].imshow(ov_crop); axes[i,1].set_title(f'CENTER CROP: {n_crop} lines', fontsize=9); axes[i,1].axis('off')
plt.suptitle('Lane detection: full panorama vs central single-camera crop', fontsize=12)
plt.tight_layout()
plt.savefig('outputs/lane_panorama_vs_crop.png', dpi=110, bbox_inches='tight')
plt.show()